# Efficiency plotting 

Copied most functionality from Moon's https://github.com/wjdanswjddl/cafpyana/blob/release/numucc_1p0pi/analysis_village/numucc_1p0pi/notebooks/event_selection.ipynb

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# print avaialbe memory
import psutil
print(psutil.virtual_memory())

svmem(total=506987724800, available=443870363648, percent=12.4, used=63117361152, free=277471490048, active=88875737088, inactive=130751504384, buffers=7815168, cached=170715398144, shared=88264704, slab=6370865152)


In [3]:
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from os import makedirs, path
from datetime import datetime
import pickle

os.environ['BEARER_TOKEN_FILE'] = f"/tmp/bt_u{os.getuid()}_sbnd"

# local imports
# sys.path.append('../../../')
cwd = Path.cwd().resolve()
# repo_root = next((candidate for candidate in [cwd, *cwd.parents] if (candidate / 'analysis_village').exists()), None)
# if repo_root is None:
#     raise RuntimeError('Could not locate the repository root from the current notebook working directory')
repo_root = Path('/nashome/m/micarrig/sbnd/nueCCNp/cafpyana')
sys.path.append(str(repo_root))
from analysis_village.nueNp0Pi.config.plots import VariableConfig
from analysis_village.nueNp0Pi.config.settings import *
from analysis_village.nueNp0Pi.selections import *
from analysis_village.nueNp0Pi.utils import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use(repo_root / 'analysis_village' / 'nueNp0Pi' / 'notebooks' / 'presentation.mplstyle')

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings('ignore', category=RuntimeWarning)

os.environ['CAFPYANA_LOG_LEVEL'] = 'DEBUG'
import os

#TODO change this when systematics are implemented (also change naming)
os.environ["NUMUCC_SYST_DISK_ROOT"] = ""


In [7]:
from analysis_village.nueNp0Pi.event_selection import build_event_selection_pipeline
from pyanalib.event_selection_pipeline import EventSelectionPipelineConfig
from analysis_village.nueNp0Pi.config.datasets import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
syst_tag = ""

# Optional overrides (None -> dated work dir under /exp/sbnd/data/users/$USER/...)
batch_work_base = f"/exp/sbnd/data/users/micarrig/nueNp0Pi/{today_str}"
# batch_work_base = f'/Users/micarrig/Desktop/SBND/cafpyana/plots/'
# batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-{syst_tag}-{today_str}")
batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-debug")

batch_cfg = EventSelectionPipelineConfig(
    work_base=batch_work_base,
    plots_dir=batch_plots_dir,
    max_job_bytes=int(10.0 * 1024**3),  # 1 GiB per job
    mc_univ_syst=(), #("Flux", "G4", "GENIE"),
    skip_existing_batches=False,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    # max_files_per_sample=1,  # smoke test: one file per sample
)
pipeline = build_event_selection_pipeline(batch_cfg)

records, jobs, manifest_path = pipeline.discover_jobs()
print(f"Batched workflow: {len(jobs)} job(s)  manifest={manifest_path}")
for j in jobs[:8]:
    print(f"  {j.sample} {j.tag}: {len(j.files)} file(s), {j.total_bytes / (1024**3):.3f} GiB")
if len(jobs) > 8:
    print(f"  ... and {len(jobs) - 8} more")


[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df
[event_selection] mc: found 6 file(s)
[event_selection] sample=mc  files=6  total=15.90 GiB
[event_selection] 2 job(s) under size budget
  mc batch_0000: 3 file(s), 7.863 GiB
  mc batch_0001: 3 file(s), 8.041 GiB
Batched workflow: 2 job(s)  manifest=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260820/manifest.json
  mc batch_0000: 3 file(s), 7.863 GiB
  mc batch_0001: 3 file(s), 8.041 GiB


### Stacked event-distribution breakdown plots

By default, `build_pipeline()` (in `config/stages.py`) attaches a stacked breakdown plot
for every `EFFICIENCY_VARS` variable, by topology, at the final selection stage -- no
changes needed below to get them. Set the `EVT_BREAKDOWN_*` attributes on the
`config.stages` module before running the pipeline (cell below) to change the
breakdown type, which stage(s) they're attached to, or which variables get one.

In [8]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)

[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df


[event_selection] mc: found 6 file(s)
[event_selection] sample=mc  files=6  total=15.90 GiB
[event_selection] 2 job(s) under size budget
  mc batch_0000: 3 file(s), 7.863 GiB
  mc batch_0001: 3 file(s), 8.041 GiB
[event_selection] WORK_BASE=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260820
[event_selection] BATCHES_DIR=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260820/batches
[event_selection] sample=mc batch_0000  files=3  size=7.863 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0000  n_evt=1535802  pot=1.128e+19
[event_selection] sample=mc batch_0001  files=3  size=8.041 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

...

[aggregate] sample=mc -> 2 chunks
[aggregate] sample=data -> 0 chunks
[aggregate] sample=intime -> 0 chunks
[aggregate] sample=offbeam -> 0 chunks
[aggregate] sample=dirt -> 0 chunks
[aggregate] aggregating 2 chunks for sample=mc
[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=2.256e+19 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=2.256e+19 -> POT label=2.26$\times 10^{19}$
[aggregate] systematics disk root: None


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] overlay plots without syst covariance (34 plots, 34 distinct var_save_name): E_nu, electron-dedx, electron-e, electron-e-res, electron-primary-score, electron-softmax-score, electron-vertex-distance, num-electrons, num-muons, num-photons, num-pions, num-protons, opening_angle, opening_angle-res, opening_angle_beam, opening_angle_beam-res, particle_ke, proton-p, proton-p-res, proton-softmax-score, secondary-proton-p, secondary-proton-p-res, tki-del_Tp, tki-del_Tp-res, tki-del_Tp_lp, tki-del_Tp_lp-res, tki-del_alpha, tki-del_alpha-res, tki-del_alpha_lp, tki-del_alpha_lp-res, tki-del_phi, tki-del_phi-res, tki-del_phi_lp, tki-del_phi_lp-res


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] final selection purity: 86.29% (weighted)  86.29% (raw counts, notebook-style)
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection-debug/pkl/eff_dict.pkl
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection-debug/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection-debug
Batched workflow complete.
  batches: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260820/batches
  plots  : /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection-debug
  manifest: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260820/manifest.json
  POT    : 2.26$\times 10^{19}$


In [150]:
import analysis_village.nueNp0Pi.config.stages as stages_mod

# Uncomment/edit any of these before running the pipeline below. Defaults shown.
stages_mod.EVT_BREAKDOWN_TYPE = "genie"           # "topology" | "genie" | "pdg"
stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"
# stages_mod.EVT_BREAKDOWN_STAGE_KEYS = None            # None -> final stage only; or e.g. ["no_photons", "electron_dedx"]
# stages_mod.EVT_BREAKDOWN_VARS = stages_mod.EFFICIENCY_VARS  # or e.g. [VariableConfig.neutrino_energy()]
# stages_mod.EVT_BREAKDOWN_VARS = []                    # uncomment to disable these plots entirely

# Output subdirectory for these plots (under the plots dir), default "selection". Set this
# alongside EVT_BREAKDOWN_TYPE if you want to run more than one breakdown_type (e.g. once as
# "topology", once as "genie") without the later run's selection_<var>.png files
# overwriting the earlier run's -- e.g.:
# stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"


In [151]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)


[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_v3/*.df
[event_selection] mc: found 160 file(s)
[event_selection] sample=mc  files=160  total=426.49 GiB
[event_selection] 53 job(s) under size budget
  mc batch_0000: 3 file(s), 7.863 GiB
  mc batch_0001: 3 file(s), 8.041 GiB
  mc batch_0002: 3 file(s), 8.054 GiB
  mc batch_0003: 4 file(s), 9.650 GiB
  mc batch_0004: 3 file(s), 7.987 GiB
  mc batch_0005: 3 file(s), 7.971 GiB
  mc batch_0006: 3 file(s), 7.952 GiB
  mc batch_0007: 4 file(s), 9.630 GiB
  mc batch_0008: 3 file(s), 7.450 GiB
  mc batch_0009: 3 file(s), 7.936 GiB
  mc batch_0010: 3 file(s), 8.067 GiB
  mc batch_0011: 3 file(s), 8.101 GiB
  ... and 41 more
[event_selection] WORK_BASE=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260820
[event_selection] BATCHES_DIR=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260820/batches
[event_selection] sample=mc batch_0000  files=3  size=7.863 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] precomputed N-1 cut masks for stages: ['electron_dedx', 'electron_primary', 'electron_softmax', 'electron_vertex_distance', 'good_electron', 'good_proton', 'is_contained', 'is_fiducial', 'is_flash_matched', 'no_muons', 'no_photons', 'no_pions', 'proton_softmax']
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] >>> stage='allreco' plots=0 breakdown=True eff=True
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     bar breakdown …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     efficiency accumulators …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] <<< stage='allreco' finished
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] >>> stage='is_fiducial' plots=0 breakdown=True eff=True
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     executing cut '_cut' …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     after cut: len(evt)=138179 len(trk)=1490936
[DEBUG] cafpyana.pyanalib.c

[event_selection] done sample=mc batch_0000  n_evt=1535802  pot=1.128e+19
[event_selection] sample=mc batch_0001  files=3  size=8.041 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

...

[event_selection] done sample=mc batch_0030  n_evt=1570994  pot=1.128e+19
[event_selection] sample=mc batch_0031  files=3  size=8.084 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0031  n_evt=1578189  pot=1.127e+19
[event_selection] sample=mc batch_0032  files=3  size=8.137 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

KeyboardInterrupt: 

In [ ]:
import analysis_village.nueNp0Pi.config.stages as stages_mod
from analysis_village.nueNp0Pi.config.plots import VariableConfig

stages_mod.EVT_BREAKDOWN_TYPE = "topology"           # "topology" | "genie" | "pdg"
stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"

stages_mod.DISABLE_EFFICIENCY_ACCUMULATION = True

stages_mod.N_MINUS_1_STAGE_KEYS = [
    "no_muons",
    "no_pions",
    "no_photons",
    "good_electron",
    "good_proton",
    "electron_softmax", 
    "electron_primary",
    "proton_softmax",
    "electron_vertex_distance",
    "electron_dedx"]
stages_mod.N_MINUS_1_VARS = {
    "no_muons": [VariableConfig.num_muons(), VariableConfig.leading_muon_ke()],
    "no_pions": [VariableConfig.num_pions(), VariableConfig.leading_pion_ke()],
    "no_photons": [VariableConfig.num_photons()], #VariableConfig.leading_photon_ke()],
    "good_electron": [VariableConfig.num_electrons(), VariableConfig.electron_energy()],
    "good_proton": [VariableConfig.num_protons(), VariableConfig.leading_proton_ke(), VariableConfig.proton_momentum()],
    "electron_softmax": [VariableConfig.electron_softmax_score()],
    "electron_primary": [VariableConfig.electron_primary_score()],
    "proton_softmax": [VariableConfig.proton_softmax_score()],
    "electron_vertex_distance": [VariableConfig.electron_vertex_distance()],
    "electron_dedx": [VariableConfig.electron_dedx()],
}

pipeline = build_event_selection_pipeline(batch_cfg)
batch_result = pipeline.run_full()


[event_selection] mc: looking in /Users/micarrig/Desktop/SBND/data/*.df
[event_selection] mc: found 1 file(s)
[event_selection] sample=mc  files=1  total=2.44 GiB
[event_selection] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 2.435 GiB
[event_selection] WORK_BASE=/Users/micarrig/Desktop/SBND/cafpyana/plots
[event_selection] BATCHES_DIR=/Users/micarrig/Desktop/SBND/cafpyana/plots/batches
[event_selection] sample=mc batch_0000  files=1  size=2.435 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0000  n_evt=507555  pot=3.762e+18
[event_selection] sample=mc wrote 1 batch pickle(s)


[INFO] cafpyana.pyanalib.chunked_selection: aggregating 1 chunk file(s)
[INFO] cafpyana.pyanalib.chunked_selection: merging samples: ['mc']
[INFO] cafpyana.pyanalib.chunked_selection: applied global exposure scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}


[aggregate] sample=mc -> 1 chunks
[aggregate] sample=data -> 0 chunks
[aggregate] sample=intime -> 0 chunks
[aggregate] sample=offbeam -> 0 chunks
[aggregate] sample=dirt -> 0 chunks
[aggregate] aggregating 1 chunks for sample=mc
[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=3.762e+18 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=3.762e+18 -> POT label=3.76$\times 10^{18}$
[aggregate] systematics disk root: None


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] overlay plots without syst covariance (49 plots, 37 distinct var_save_name): E_nu, electron-dedx, electron-e, electron-e-res, electron-primary-score, electron-softmax-score, electron-vertex-distance, leading_muon_ke, leading_pion_ke, leading_proton_ke, num-electrons, num-muons, num-photons, num-pions, num-protons, opening_angle, opening_angle-res, opening_angle_beam, opening_angle_beam-res, particle_ke, proton-p, proton-p-res, proton-softmax-score, secondary-proton-p, secondary-proton-p-res, tki-del_Tp, tki-del_Tp-res, tki-del_Tp_lp, tki-del_Tp_lp-res, tki-del_alpha, tki-del_alpha-res, tki-del_alpha_lp, tki-del_alpha_lp-res, tki-del_phi, tki-del_phi-res, tki-del_phi_lp, tki-del_phi_lp-res


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] no efficiency data -> skipping
[aggregate] wrote /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260817/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /Users/micarrig/Desktop/SBND/cafpyana/plots/event_selection--20260817


# Debug

In [9]:
import uproot

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [10]:
f_medulla = uproot.open("/exp/sbnd/app/users/micarrig/nueCCNp/output_nueCCNp_mc_5e18.root:events/signal/all", library="pd")

df_medulla = f_medulla.arrays(library="pd")

hdf_path = '/nashome/m/micarrig/sbnd/nueCCNp/cafpyana/test.df'
key = 'evt'

with pd.HDFStore(hdf_path, mode='r') as store:
    chunks = sorted(k for k in store.keys() if k.strip('/').startswith(key))
    print(f"Found {len(chunks)} chunks for key '{key}': {chunks}")
    df = pd.concat([store[k] for k in chunks])

Found 1 chunks for key 'evt': ['/evt_0']


In [6]:
df_medulla

KeyboardInterrupt: 

In [7]:
df

KeyboardInterrupt: 

In [11]:
# medulla signal events

signal_mask = (df_medulla['true_category'] == 0) | (df_medulla['true_category'] == 1)

df_medulla.loc[signal_mask, ['Run', 'Subrun', 'Evt']]


,Run,Subrun,Evt
1776,3,86,37
7283,4,75,88
16228,7,57,38
18403,8,49,98
21709,9,76,48
37216,15,11,2
59343,22,52,68
60266,22,79,89
62300,23,20,69
84673,30,52,66


In [12]:
# cafpyana signal events

signal_mask = (df.rec.dlp_true['true_signal1p'] == True) | (df.rec.dlp_true['true_signalNp'] == True)

df.loc[signal_mask, ['run', 'subrun', 'evt']]


run subrun evt
                                            
                                            
                                            
                                            
__ntuple entry rec.dlp..index               
0        145   0                3     86  37
         599   1                4     75  88
         1334  2                7     57  38
         1519  4                8     49  98
         1781  1                9     76  48
         3037  0               15     11   2
         4829  0               22     52  68
         4902  0               22     79  89
         5067  1               23     20  69
         7061  8               30     94  52

In [139]:
pd.concat([
    df.loc[signal_mask, ['run', 'subrun', 'evt']],
    df.loc[signal_mask].rec.dlp_true[['proton_energy_true', 'ele_energy_true', 'muon_energy_true', 'pion_energy_true', 'photon_energy_true', 'pion_mask_true', 'photon_mask_true', 'true_signal1p', 'true_signalNp']]
], axis=1)


(run, , , , )  (subrun, , , , )  (evt, , , , )  \
__ntuple entry rec.dlp..index                                                   
0        145   0                           3                86             37   
         599   1                           4                75             88   
         1334  2                           7                57             38   
         1519  4                           8                49             98   
         1781  1                           9                76             48   
         3037  0                          15                11              2   
         3153  1                          15                78             78   
         3761  5                          18                11             49   
               8                          18                11             49   
         4829  0                          22                52             68   
         4902  0                          22                79             89   
         5067  1                          23                20             69   
         5684  0                          25                75             23   
         7061  8                          30                94             52   

                               (proton_energy_true, , )  \
__ntuple entry rec.dlp..index                             
0        145   0                             398.448205   
         599   1                             281.212945   
         1334  2                             139.515302   
         1519  4                              40.591564   
         1781  1                             122.200287   
         3037  0                              62.066490   
         3153  1                             637.589779   
         3761  5                              56.307022   
               8                              56.307022   
         4829  0                              86.648638   
         4902  0                             318.260611   
         5067  1                             225.633544   
         5684  0                             530.166712   
         7061  8                             273.176140   

                               (ele_energy_true, , )  (muon_energy_true, , )  \
__ntuple entry rec.dlp..index                                                  
0        145   0                         2097.166883                     NaN   
         599   1                          846.942876                     NaN   
         1334  2                          528.199264                     NaN   
         1519  4                          696.010582                     NaN   
         1781  1                         1488.197271                     NaN   
         3037  0                          664.116087                     NaN   
         3153  1                          600.373314                     NaN   
         3761  5                         1225.419225                     NaN   
               8                         1225.419225                     NaN   
         4829  0                          693.967585                     NaN   
         4902  0                         1625.792760                     NaN   
         5067  1                          819.915635                     NaN   
         5684  0                         1288.906048                     NaN   
         7061  8                         1269.506585                     NaN   

                               (pion_energy_true, , )  \
__ntuple entry rec.dlp..index                           
0        145   0                                  NaN   
         599   1                                  NaN   
         1334  2                                  NaN   
         1519  4                                  NaN   
         1781  1                                  NaN   
         3037  0                                  NaN   
         3153  1                                  Na

In [136]:
# medulla event type of events selected by cafpyana

keys = df.loc[signal_mask, ['run', 'subrun', 'evt']]
idx = pd.MultiIndex.from_frame(keys)
medulla_idx = pd.MultiIndex.from_arrays([df_medulla['Run'], df_medulla['Subrun'], df_medulla['Evt']])
df_medulla.loc[medulla_idx.isin(idx), ['Run', 'Subrun', 'Evt', 'true_category', 'true_leading_proton_ke', 'true_leading_proton_energy', 'true_leading_electron_energy', 'true_leading_electron_ke']]



,Run,Subrun,Evt,true_category,true_leading_proton_ke,true_leading_proton_energy,true_leading_electron_energy,true_leading_electron_ke
1776,3,86,37,1.0,398.448205,1336.720207,2097.166883,2097.166883
1777,3,86,37,10.0,NaN,NaN,42.931830,42.420759
1778,3,86,37,10.0,NaN,NaN,42.701891,42.190944
1779,3,86,37,10.0,NaN,NaN,85.238525,84.727269
1780,3,86,37,10.0,NaN,NaN,85.238525,84.727269
1781,3,86,37,8.0,75.194961,1013.466975,36.548029,36.037060
1782,3,86,37,10.0,NaN,NaN,45.595208,45.084181
1783,3,86,37,NaN,NaN,NaN,NaN,NaN
1784,3,86,37,10.0,NaN,NaN,33.782640,33.271718
1785,3,86,37,10.0,NaN,NaN,NaN,NaN


In [130]:
# cafpyana event type of events selected by medulla

keys = df_medulla.loc[signal_mask, ['Run', 'Subrun', 'Evt']]
idx = pd.MultiIndex.from_frame(keys)
cafpyana_idx = pd.MultiIndex.from_arrays([df['run'], df['subrun'], df['evt']])
evt_info = df.loc[cafpyana_idx.isin(idx), ['run', 'subrun', 'evt']]
branches = df.loc[cafpyana_idx.isin(idx)].rec.dlp_true[['true_signal1p', 'true_signalNp', 'bkgd_oofv', 'bkgd_oops', \
    'bkgd_pi', 'bkgd_mu', 'bkgd_photon', 'bkgd_nueOther', 'bkgd_numu', 'bkgd_nc', 'bkgd_other', \
    'muon_energy_true', 'ele_energy_true', 'proton_energy_true', 'muon_count_true']]

pd.concat([
    evt_info,
    branches
], axis=1)


(run, , , , )  (subrun, , , , )  (evt, , , , )  \
__ntuple entry rec.dlp..index                                                   
0        145   0                           3                86             37   
               1                           3                86             37   
               2                           3                86             37   
               3                           3                86             37   
               4                           3                86             37   
               5                           3                86             37   
               6                           3                86             37   
               7                           3                86             37   
               8                           3                86             37   
               9                           3                86             37   
               10                          3                86             37   
               11                          3                86             37   
               12                          3                86             37   
         599   0                           4                75             88   
               1                           4                75             88   
               2                           4                75             88   
               3                           4                75             88   
               4                           4                75             88   
               5                           4                75             88   
               6                           4                75             88   
               7                           4                75             88   
               8                           4                75             88   
               9                           4                75             88   
               10                          4                75             88   
               11                          4                75             88   
               12                          4                75             88   
         1334  0                           7                57             38   
               1                           7                57             38   
               2                           7                57             38   
               3                           7                57             38   
               4                           7                57             38   
               5                           7                57             38   
               6                           7                57             38   
               7                           7                57             38   
         1519  0                           8                49             98   
               1                           8                49             98   
               2                           8                49             98   
               3                           8                49             98   
               4                           8                49             98   
               5                           8                49             98   
               6                           8                49             98   
               7                           8                49             98   
               8                           8                49             98   
               9                           8                49             98   
               10                          8                49             98   
               11                          8                49             98   
         1781  0                           9                76             48   
               1                           9                76

In [63]:
signal_mask = (df_medulla['true_category'] == 0) | (df_medulla['true_category'] == 1)

df_medulla.loc[signal_mask, ['Run', 'Subrun', 'Evt', 'true_leading_muon_ke', 'true_leading_muon_energy']]

,Run,Subrun,Evt,true_leading_muon_ke,true_leading_muon_energy
1776,3,86,37,NaN,NaN
7283,4,75,88,NaN,NaN
16228,7,57,38,NaN,NaN
18403,8,49,98,NaN,NaN
21709,9,76,48,NaN,NaN
37216,15,11,2,NaN,NaN
59343,22,52,68,NaN,NaN
60266,22,79,89,NaN,NaN
62300,23,20,69,NaN,NaN
84673,30,52,66,NaN,NaN


In [34]:
df_medulla[(df_medulla['Run'] == 4) & (df_medulla['Evt'] == 88) & (df_medulla['Subrun'] == 75)]

,reco_containment_cut,reco_cut_type,reco_dalphaT,reco_dpL,reco_dpL_lp,reco_dpT,reco_dpT_lp,reco_dphiT,reco_ele_beam_open_angle,reco_electron_multiplicity,reco_electron_softmax_cut,reco_fiducial_cut,reco_flash_cut,reco_flash_time,reco_has_electron,reco_has_proton,reco_is_data,reco_is_nu,reco_leading_ele_energy_cut,reco_leading_electron_axial_cut,reco_leading_electron_axial_spread,reco_leading_electron_azimuthal_angle,reco_leading_electron_calo_ke,reco_leading_electron_containment_cut,reco_leading_electron_csda_ke,reco_leading_electron_dedx,reco_leading_electron_directional_spread,reco_leading_electron_electron_softmax,reco_leading_electron_end_dir_x,reco_leading_electron_end_dir_y,reco_leading_electron_end_dir_z,reco_leading_electron_end_x,reco_leading_electron_end_y,reco_leading_electron_end_z,reco_leading_electron_energy,reco_leading_electron_ke,reco_leading_electron_length,reco_leading_electron_mcs_ke,reco_leading_electron_mip_softmax,reco_leading_electron_muon_softmax,reco_leading_electron_p,reco_leading_electron_photon_softmax,reco_leading_electron_pion_softmax,reco_leading_electron_polar_angle,reco_leading_electron_primary_softmax,reco_leading_electron_proton_softmax,reco_leading_electron_secondary_softmax,reco_leading_electron_start_dedx,reco_leading_electron_start_dir_x,reco_leading_electron_start_dir_y,reco_leading_electron_start_dir_z,reco_leading_electron_start_x,reco_leading_electron_start_y,reco_leading_electron_start_z,reco_leading_electron_vertex_distance,reco_leading_muon_azimuthal_angle,reco_leading_muon_energy,reco_leading_muon_ke,reco_leading_muon_muon_softmax,reco_leading_muon_p,reco_leading_muon_polar_angle,reco_leading_muon_primary_softmax,reco_leading_muon_start_dedx,reco_leading_photon_azimuthal_angle,reco_leading_photon_energy,reco_leading_photon_p,reco_leading_photon_photon_softmax,reco_leading_photon_polar_angle,reco_leading_photon_primary_softmax,reco_leading_photon_start_dedx,reco_leading_pion_azimuthal_angle,reco_leading_pion_energy,reco_leading_pion_ke,reco_leading_pion_p,reco_leading_pion_pion_softmax,reco_leading_pion_polar_angle,reco_leading_pion_primary_softmax,reco_leading_pion_start_dedx,reco_leading_proton_azimuthal_angle,reco_leading_proton_energy,reco_leading_proton_ke,reco_leading_proton_muon_softmax,reco_leading_proton_p,reco_leading_proton_pion_softmax,reco_leading_proton_polar_angle,reco_leading_proton_primary_softmax,reco_leading_proton_proton_softmax,reco_leading_proton_start_dedx,reco_leading_proton_vertex_distance,reco_muon_multiplicity,reco_no_charged_pions,reco_no_cut,reco_no_muons,reco_no_photons,reco_no_protons,reco_nonelectron_containment_cut,reco_opening_angle,reco_photon_multiplicity,reco_photon_multiplicity25,reco_pion_multiplicity,reco_pn,reco_pn_lp,reco_primary_softmax_cut,reco_proton_multiplicity,reco_proton_softmax_cut,reco_reco_interaction_type,reco_single_tpc_contained,reco_track_length,reco_two_photon_candidates,reco_valid_flashmatch,reco_vertex_distance_cut,reco_visible_energy,true_baseline,true_category,true_cc,true_containment_cut,true_cut_type,true_dalphaT,true_dpL,true_dpL_lp,true_dpT,true_dpT_lp,true_dphiT,true_ele_beam_open_angle,true_electron_multiplicity,true_fiducial_cut,true_flash_cut,true_hadronic_invariant_mass,true_has_electron,true_has_proton,true_interaction_mode,true_interaction_type,true_is_data,true_is_nu,true_leading_ele_energy_cut,true_leading_electron_azimuthal_angle,true_leading_electron_calo_ke,true_leading_electron_containment_cut,true_leading_electron_csda_ke,true_leading_electron_end_dir_x,true_leading_electron_end_dir_y,true_leading_electron_end_dir_z,true_leading_electron_end_x,true_leading_electron_end_y,true_leading_electron_end_z,true_leading_electron_energy,true_leading_electron_ke,true_leading_electron_length,true_leading_electron_mcs_ke,true_leading_electron_p,true_leading_electron_polar_angle,true_leading_electron_start_dir_x,true_leading_electron_start_dir_y,true_leading_electron_start_dir_z,true_leading_electron_start_x,true_lea

In [115]:
fin = uproot.open("root://fndcadoor.fnal.gov:/sbnd/persistent/users/mueller/MCP2025B/simulation/mc5e18/input000-reweighted.flat.caf.root:recTree")

In [13]:
fin.keys()

['rec.crt_hits..length',
 'rec.crt_hits.pe',
 'rec.crt_hits.plane',
 'rec.crt_hits.position.x',
 'rec.crt_hits.position.y',
 'rec.crt_hits.position.z',
 'rec.crt_hits.position_err.x',
 'rec.crt_hits.position_err.y',
 'rec.crt_hits.position_err.z',
 'rec.crt_hits.t0',
 'rec.crt_hits.t1',
 'rec.crt_hits.time',
 'rec.crt_spacepoints..length',
 'rec.crt_spacepoints.complete',
 'rec.crt_spacepoints.pe',
 'rec.crt_spacepoints.position.x',
 'rec.crt_spacepoints.position.y',
 'rec.crt_spacepoints.position.z',
 'rec.crt_spacepoints.position_err.x',
 'rec.crt_spacepoints.position_err.y',
 'rec.crt_spacepoints.position_err.z',
 'rec.crt_spacepoints.time',
 'rec.crt_spacepoints.time_err',
 'rec.crt_tracks..length',
 'rec.crt_tracks.hita.pe',
 'rec.crt_tracks.hita.plane',
 'rec.crt_tracks.hita.position.x',
 'rec.crt_tracks.hita.position.y',
 'rec.crt_tracks.hita.position.z',
 'rec.crt_tracks.hita.position_err.x',
 'rec.crt_tracks.hita.position_err.y',
 'rec.crt_tracks.hita.position_err.z',
 'rec.cr

In [116]:
arrays = fin.arrays(["rec.hdr.evt", "rec.hdr.run", "rec.dlp_true.particles.ke", \
                    "rec.dlp_true.particles.pdg_code", "rec.dlp_true.particles.is_primary", \
                    "rec.dlp_true.particles.pid", "rec.dlp_true.particles.is_valid", \
                    "rec.dlp_true.particles.is_contained", "rec.dlp_true.particles.is_matched", \
                    "rec.dlp_true.particles.match_overlaps", "rec.dlp_true.id"], \
        "(rec.hdr.run == 4) & (rec.hdr.evt == 88) & (rec.hdr.subrun == 75)", \
        library="ak")

In [36]:
arrays

<Array [{'rec.hdr.evt': 88, ...}] type='1 * {"rec.hdr.evt": uint32, "rec.hd...'>

In [103]:
# ak.max(arrays['rec.dlp_true.particles.ke'][abs(arrays['rec.dlp_true.particles.pdg_code']) == 13], axis=1)
arrays['rec.dlp_true.particles.ke'][(abs(arrays['rec.dlp_true.particles.pdg_code']) == 13) & (arrays['rec.dlp_true.particles.is_primary'] == True)]

muon_mask = (abs(arrays['rec.dlp_true.particles.pdg_code']) == 13) 
primary_mask = (arrays['rec.dlp_true.particles.is_primary'] == True)
electron_mask = (abs(arrays['rec.dlp_true.particles.pdg_code']) == 11)

total_particles = ak.count(arrays['rec.dlp_true.particles.ke'], axis=1)
total_muons = ak.count(arrays['rec.dlp_true.particles.ke'][muon_mask], axis=1)
primary_particles = ak.count(arrays['rec.dlp_true.particles.ke'][primary_mask], axis=1)
primary_muons = ak.count(arrays['rec.dlp_true.particles.ke'][muon_mask & primary_mask], axis=1)
total_electrons = ak.count(arrays['rec.dlp_true.particles.ke'][electron_mask], axis=1)
primary_electrons = ak.count(arrays['rec.dlp_true.particles.ke'][electron_mask & primary_mask], axis=1)

print(f"Total particles: {total_particles}")
print(f"Total muons: {total_muons}")
print(f"Primary particles: {primary_particles}")
print(f"Primary muons: {primary_muons}")
print(f"Total electrons: {total_electrons}")
print(f"Primary electrons: {primary_electrons}")

Total particles: [44]
Total muons: [12]
Primary particles: [14]
Primary muons: [12]
Total electrons: [31]
Primary electrons: [1]


In [104]:
ak.where((abs(arrays['rec.dlp_true.particles.pdg_code']) == 11) & (arrays['rec.dlp_true.particles.is_primary'] == True))

(<Array [0] type='1 * int64'>, <Array [0] type='1 * int64'>)

In [105]:
arrays['rec.dlp_true.particles.ke'][arrays['rec.dlp_true.particles.is_primary'] == True].to_list()

[[846.9428758010413,
  281.21294547112575,
  1792.3799529605703,
  728.0755741048146,
  40559.422251045915,
  6784.1842041569,
  41714.837066202905,
  3183.0886892797903,
  15751.461948761164,
  1812.8190860038826,
  1039.7338316419682,
  1987.5710658618684,
  10714.748641464237,
  4121.800440904377]]

In [125]:
primary_mask = arrays['rec.dlp_true.particles.is_primary'] == True
valid_mask = arrays['rec.dlp_true.particles.is_valid'] == True
# first_int, _ = ak.broadcast_arrays((arrays['rec.dlp_true.id'] == 0), valid_mask)

first_int = arrays['rec.dlp_true.id'] == 0

In [128]:
arrays['rec.dlp_true.particles.pdg_code'][0, :].to_list()

[11,
 2212,
 13,
 11,
 -13,
 13,
 -13,
 11,
 11,
 11,
 11,
 13,
 11,
 11,
 11,
 11,
 11,
 -11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 13,
 11,
 -13,
 13,
 11,
 13,
 11,
 11,
 11,
 13,
 11,
 13,
 11,
 11,
 -13,
 11,
 11,
 11,
 11]

In [118]:
arrays['rec.dlp_true.id'].to_list()

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]]

In [109]:
arrays['rec.dlp_true.particles.ke'][mask].to_list()

[[846.9428758010413,
  281.21294547112575,
  1792.3799529605703,
  40559.422251045915,
  6784.1842041569,
  41714.837066202905,
  3183.0886892797903,
  15751.461948761164,
  1812.8190860038826,
  1039.7338316419682,
  1987.5710658618684,
  10714.748641464237,
  4121.800440904377]]

In [112]:
import uproot
from makedf.makedf import make_spine_int_df, make_spine_part_df
from makedf.branches import spinetpart_branches
from pyanalib.pandas_helpers import loadbranches, pad_column_name, rename_to_XYZ

RUN, SUBRUN, EVT = 4, 75, 88

CAF = "root://fndcadoor.fnal.gov:/sbnd/persistent/users/mueller/MCP2025B/simulation/mc5e18/input000-reweighted.flat.caf.root"

with uproot.open(CAF) as fin:
    slcdf        = make_spine_int_df(fin)
    spinetpart_df = loadbranches(fin["recTree"], spinetpart_branches)
    rename_to_XYZ(spinetpart_df, ["momentum", "end_point", "start_point", "start_dir", "end_dir", "vertex"])

# locate the entry via the df already loaded in the notebook
event_mask = (df['run'] == RUN) & (df['subrun'] == SUBRUN) & (df['evt'] == EVT)
row = df[event_mask]
print("Reco interactions for this event:")
print(row[['run','subrun','evt']].to_string())

entry    = row.index.get_level_values('entry')[0]
reco_int = row.index.get_level_values('rec.dlp..index')[0]

# which true interaction is matched to this reco slice?
true_int_col = pad_column_name(("rec.dlp_true..index",), slcdf)
matched_true_int = slcdf.loc[(entry, reco_int), true_int_col]
print(f"\nEntry={entry}, reco_int={reco_int} is matched to true_int={matched_true_int}")

# all muons in this event, showing which true interaction they belong to
pdg_col  = pad_column_name(('rec', 'dlp_true', 'particles', 'pdg_code'),   spinetpart_df)
ke_col   = pad_column_name(('rec', 'dlp_true', 'particles', 'ke'),         spinetpart_df)
prim_col = pad_column_name(('rec', 'dlp_true', 'particles', 'is_primary'), spinetpart_df)
valid_col= pad_column_name(('rec', 'dlp_true', 'particles', 'is_valid'),   spinetpart_df)

event_parts = spinetpart_df.loc[entry]
muons = event_parts[abs(event_parts[pdg_col]) == 13]
print(f"\nAll muons (index=[true_int, particle_idx]) — matched true_int is {matched_true_int}:")
print(muons[[ke_col, prim_col, valid_col]].to_string())


Reco interactions for this event:
                              run subrun evt
                                            
                                            
                                            
                                            
__ntuple entry rec.dlp..index               
0        599   0                4     75  88
               1                4     75  88
               2                4     75  88
               3                4     75  88
               4                4     75  88
               5                4     75  88
               6                4     75  88
               7                4     75  88
               8                4     75  88
               9                4     75  88
               10               4     75  88
               11               4     75  88
               12               4     75  88

Entry=599, reco_int=0 is matched to true_int=nan

All muons (index=[true_int, particle_idx]) — matched true_in

In [113]:
RUN, SUBRUN, EVT = 4, 75, 88

# Get only the row(s) that actually passed the signal selection
sig_col1 = pad_column_name(('rec', 'dlp_true', 'true_signal1p'), df)
sig_col2 = pad_column_name(('rec', 'dlp_true', 'true_signalNp'), df)
event_mask = (df['run'] == RUN) & (df['subrun'] == SUBRUN) & (df['evt'] == EVT)
sig_mask   = event_mask & ((df[sig_col1] == True) | (df[sig_col2] == True))

row = df[sig_mask]
print("Signal reco interaction(s):")
print(row[['run','subrun','evt']].to_string())

entry    = row.index.get_level_values('entry')[0]
reco_int = row.index.get_level_values('rec.dlp..index')[0]

# which true interaction is matched to THIS reco slice?
true_int_col     = pad_column_name(("rec.dlp_true..index",), slcdf)
matched_true_int = slcdf.loc[(entry, reco_int), true_int_col]
print(f"\nEntry={entry}, reco_int={reco_int} → matched true_int={matched_true_int}")

# muons in this event — is any of them under matched_true_int?
event_parts = spinetpart_df.loc[entry]
muons = event_parts[abs(event_parts[pdg_col]) == 13]
print(f"\nAll muons (matched_true_int={matched_true_int}):")
print(muons[[ke_col, prim_col, valid_col]].to_string())


Signal reco interaction(s):
                              run subrun evt
                                            
                                            
                                            
                                            
__ntuple entry rec.dlp..index               
0        599   1                4     75  88

Entry=599, reco_int=1 → matched true_int=0.0

All muons (matched_true_int=0.0):
                                                            rec                    
                                                       dlp_true                    
                                                      particles                    
                                                             ke is_primary is_valid
                                                                                   
rec.dlp_true..index rec.dlp_true.particles..index                                  
1                   0                               1792.379953         

In [26]:
# count each category in medulla and cafpyana
categories = ['true_signal1p', 'true_signalNp', 'bkgd_oofv', 'bkgd_oops', \
    'bkgd_pi', 'bkgd_mu', 'bkgd_photon', 'bkgd_nueOther', 'bkgd_numu', 'bkgd_nc', 'bkgd_other']

for i in range(len(categories)):
    index = i
    if i==0: index=1
    elif i==1: index=0
    signal_mask = (df_medulla['true_category'] == index).sum()
    print(f"Medulla: {categories[i]}: {signal_mask}")

print("Medulla: Total: ", len(df_medulla), " matched interactions: ", (df_medulla['true_category'].count()))
print('-----------------------')

for cat in categories:
    signal_mask = (df.rec.dlp_true[cat] == True).sum()
    print(f"Cafpyana: {cat}: {signal_mask}")

print("Cafpyana: Total: ", (df.rec.dlp_true[categories].sum(axis=1) > 0).sum(), " matched interactions: ", df.rec.dlp_true.is_fiducial.notnull().sum())


Medulla: true_signal1p: 9
Medulla: true_signalNp: 2
Medulla: bkgd_oofv: 6
Medulla: bkgd_oops: 12
Medulla: bkgd_pi: 5
Medulla: bkgd_mu: 0
Medulla: bkgd_photon: 1
Medulla: bkgd_nueOther: 22
Medulla: bkgd_numu: 8666
Medulla: bkgd_nc: 2288
Medulla: bkgd_other: 82493
Medulla: Total:  106786  matched interactions:  93504
-----------------------
Cafpyana: true_signal1p: 8
Cafpyana: true_signalNp: 2
Cafpyana: bkgd_oofv: 7
Cafpyana: bkgd_oops: 12
Cafpyana: bkgd_pi: 5
Cafpyana: bkgd_mu: 0
Cafpyana: bkgd_photon: 1
Cafpyana: bkgd_nueOther: 22
Cafpyana: bkgd_numu: 8666
Cafpyana: bkgd_nc: 2288
Cafpyana: bkgd_other: 95775
Cafpyana: Total:  106786  matched interactions:  93504
